In [2]:
import os

# 1. Set your token environment variable
os.environ["KAGGLE_API_TOKEN"] = "KGAT_16772a49a145b93545707940cc4e3341"

# 2. Install kagglehub package
!pip install -q kagglehub

import kagglehub

# 3. Download the competition dataset
print("Downloading dataset using your token...")
path = kagglehub.competition_download('aptos2019-blindness-detection')

print("Dataset downloaded successfully to:", path)

Dataset downloaded successfully to: /kaggle/input/competitions/aptos2019-blindness-detection


In [6]:
import os
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split
import kagglehub

# 1. Set your Kaggle API token and download dataset
os.environ["KAGGLE_API_TOKEN"] = "KGAT_16772a49a145b93545707940cc4e3341"
print("Downloading / loading dataset from cache...")
base_path = kagglehub.competition_download('aptos2019-blindness-detection')

csv_path = os.path.join(base_path, 'train.csv')
images_dir = os.path.join(base_path, 'train_images')

# 2. Organize images if not already done
if not os.path.exists('data/train'):
    print("Organizing dataset into train/val folders...")
    df = pd.read_csv(csv_path)
    df['file_name'] = df['id_code'].astype(str) + '.png'

    train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['diagnosis'])

    def organize_images(subset_df, subset_name):
        target_base = f'data/{subset_name}'
        os.makedirs(target_base, exist_ok=True)
        for _, row in subset_df.iterrows():
            class_dir = os.path.join(target_base, str(row['diagnosis']))
            os.makedirs(class_dir, exist_ok=True)
            src = os.path.join(images_dir, row['file_name'])
            dst = os.path.join(class_dir, row['file_name'])
            if os.path.exists(src):
                shutil.copy(src, dst)

    organize_images(train_df, 'train')
    organize_images(val_df, 'val')
    print("Dataset organization complete!")
else:
    print("Dataset folders already exist. Skipping organization.")

Dataset folders already exist. Skipping organization.


In [7]:
# Install required libraries
!pip install -q timm tqdm scikit-learn

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
from tqdm import tqdm

# 1. Upgraded Configurations
BATCH_SIZE = 16
EPOCHS = 15  # Increased epochs; early stopping will prevent overfitting
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-2  # Stronger weight decay for better generalization
IMAGE_SIZE = 300
NUM_CLASSES = 5
MODEL_NAME = 'efficientnet_b3'
WEIGHT_FILENAME = "hybrid_mamba_improved_best.pth"  # New unique save file

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

# 2. Advanced Data Augmentations (Crucial for medical imaging generalization)
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1), # Handles varying clinic lighting
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)), # Zoom & shift invariance
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder('data/train', transform=train_transforms)
val_dataset = datasets.ImageFolder('data/val', transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# 3. Define Upgraded Hybrid CNN-Mamba Model with Enhanced Dropout
class HybridEfficientNetMambaUpgraded(nn.Module):
    def __init__(self, num_classes=5):
        super(HybridEfficientNetMambaUpgraded, self).__init__()

        self.backbone = timm.create_model('efficientnet_b3', pretrained=True, features_only=True)

        dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE)
        features = self.backbone(dummy_input)
        final_channels = features[-1].shape[1]

        self.proj = nn.Conv2d(final_channels, 256, kernel_size=1)

        # Sequence mixing components with balanced dropout
        self.depthwise_conv = nn.Conv1d(256, 256, kernel_size=3, padding=1, groups=256)
        self.fc1 = nn.Linear(256, 512)
        self.fc2 = nn.Linear(512, 256)
        self.dropout = nn.Dropout(0.3)  # Increased dropout for regularization

        # Classification Head
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        feat_list = self.backbone(x)
        x = feat_list[-1]

        x = self.proj(x)
        b, c, h, w = x.shape

        x = x.view(b, c, h * w)
        x_res = x

        x = self.depthwise_conv(x)
        x = torch.nn.functional.silu(x)

        x = x.permute(0, 2, 1)
        x = self.fc1(x)
        x = torch.nn.functional.silu(x)
        x = self.dropout(x)
        x = self.fc2(x)

        x = x.permute(0, 2, 1)
        x = x + x_res

        out = self.classifier(x)
        return out

# Initialize Model, Optimizer, Scheduler, and Early Stopping
model = HybridEfficientNetMambaUpgraded(num_classes=NUM_CLASSES).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

class EarlyStopping:
    def __init__(self, patience=4, delta=0.0005):
        self.patience = patience
        self.delta = delta
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter = 0
            return True
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
            return False

early_stopping = EarlyStopping(patience=4)
best_val_loss = float('inf')

print(f"\n--- Starting Upgraded Hybrid Training (Saving to: {WEIGHT_FILENAME}) ---")

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0

    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{EPOCHS}]")
    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        loop.set_postfix(loss=loss.item())

    # Validation Phase
    model.eval()
    val_loss = 0.0
    val_correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels.data)

    epoch_val_loss = val_loss / len(val_dataset)
    epoch_val_acc = val_correct.double() / len(val_dataset)

    # Step the learning rate scheduler
    scheduler.step()

    print(f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

    # Check if this is the best model based on validation loss
    if early_stopping(epoch_val_loss):
        torch.save(model.state_dict(), WEIGHT_FILENAME)
        print(f"--> Saved Improved Best Model to {WEIGHT_FILENAME}")

    if early_stopping.early_stop:
        print("\nEarly stopping triggered! Training halted to prevent overfitting.")
        break

print("Training finished successfully! You can now evaluate your new model using 'hybrid_mamba_improved_best.pth'.")

Training on device: cuda


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.



--- Starting Upgraded Hybrid Training (Saving to: hybrid_mamba_improved_best.pth) ---


Epoch [1/15]: 100%|██████████| 184/184 [03:01<00:00,  1.02it/s, loss=0.758]


Val Loss: 0.5969 | Val Acc: 0.7667 | LR: 0.000099
--> Saved Improved Best Model to hybrid_mamba_improved_best.pth


...

Val Loss: 0.5046 | Val Acc: 0.8308 | LR: 0.000045


Epoch [9/15]: 100%|██████████| 184/184 [03:00<00:00,  1.02it/s, loss=2.39]  


Val Loss: 0.5019 | Val Acc: 0.8336 | LR: 0.000035


Epoch [10/15]: 100%|██████████| 184/184 [03:00<00:00,  1.02it/s, loss=0.19]  


Val Loss: 0.5563 | Val Acc: 0.8199 | LR: 0.000026

Early stopping triggered! Training halted to prevent overfitting.
Training finished successfully! You can now evaluate your new model using 'hybrid_mamba_improved_best.pth'.


In [15]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import pandas as pd
import timm
from tqdm import tqdm

# 1. Configurations
BATCH_SIZE = 16
IMAGE_SIZE = 300
NUM_CLASSES = 5
WEIGHT_PATH = "hybrid_mamba_improved_best.pth"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Automatically find test.csv and test_images path
excel_path, test_images_dir = None, None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'test.csv' in files:
        excel_path = os.path.join(root, 'test.csv')
    if 'test_images' in dirs:
        test_images_dir = os.path.join(root, 'test_images')

print(f"Found CSV: {excel_path}")
print(f"Found Images Dir: {test_images_dir}")

test_df = pd.read_csv(excel_path)
test_df['file_name'] = test_df['id_code'].astype(str) + '.png'

# 2. Blind Test Dataset Class (No labels required)
class APTOSBlindTestDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row['file_name']
        img_path = os.path.join(self.img_dir, img_name)
        
        if not os.path.exists(img_path):
            raise FileNotFoundError(f"Could not find image: {img_path}")
            
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
            
        return image, row['id_code']

test_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_dataset = APTOSBlindTestDataset(test_df, img_dir=test_images_dir, transform=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# 3. Define Model Class
class HybridEfficientNetMambaUpgraded(nn.Module):
    def __init__(self, num_classes=5):
        super(HybridEfficientNetMambaUpgraded, self).__init__()
        self.backbone = timm.create_model('efficientnet_b3', pretrained=False, features_only=True)
        
        dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE)
        features = self.backbone(dummy_input)
        final_channels = features[-1].shape[1]
        
        self.proj = nn.Conv2d(final_channels, 256, kernel_size=1)
        self.depthwise_conv = nn.Conv1d(256, 256, kernel_size=3, padding=1, groups=256)
        self.fc1 = nn.Linear(256, 512)
        self.fc2 = nn.Linear(512, 256)
        self.dropout = nn.Dropout(0.3)
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        feat_list = self.backbone(x)
        x = feat_list[-1] 
        x = self.proj(x)
        b, c, h, w = x.shape
        x = x.view(b, c, h * w) 
        x_res = x
        x = self.depthwise_conv(x)
        x = torch.nn.functional.silu(x)
        x = x.permute(0, 2, 1) 
        x = self.fc1(x)
        x = torch.nn.functional.silu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = x.permute(0, 2, 1)
        x = x + x_res
        out = self.classifier(x)
        return out

# 4. Load Model & Generate Predictions
print(f"Loading weights from {WEIGHT_PATH}...")
model = HybridEfficientNetMambaUpgraded(num_classes=NUM_CLASSES)
model.load_state_dict(torch.load(WEIGHT_PATH, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

image_ids = []
all_preds = []

with torch.no_grad():
    for images, ids in tqdm(test_loader, desc="Generating Predictions"):
        images = images.to(DEVICE)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        
        image_ids.extend(ids)
        all_preds.extend(preds.cpu().numpy())

# 5. Save Submission CSV File
submission_df = pd.DataFrame({
    'id_code': image_ids,
    'diagnosis': all_preds
})
submission_df.to_csv('submission.csv', index=False)
print("\nSuccess! Predictions saved to 'submission.csv'. Ready to submit on Kaggle.")

   

Found CSV: /kaggle/input/competitions/aptos2019-blindness-detection/test.csv
Found Images Dir: /kaggle/input/competitions/aptos2019-blindness-detection/test_images
Loading weights from hybrid_mamba_improved_best.pth...


Generating Predictions: 100%|██████████| 121/121 [01:04<00:00,  1.89it/s]


Success! Predictions saved to 'submission.csv'. Ready to submit on Kaggle.


In [16]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import pandas as pd
import timm
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

# 1. Configurations
BATCH_SIZE = 16
IMAGE_SIZE = 300
NUM_CLASSES = 5
WEIGHT_PATH = "hybrid_mamba_improved_best.pth"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Path to your labeled validation/test CSV (update filename if needed)
# Ensure this CSV has columns like 'id_code' (or filename) and 'diagnosis' (or level/target)
csv_path = '/kaggle/input/APTOS 2019 Blindness Detection/train.csv' 
# Alternatively, if you created a custom split CSV, point to it here.

# 2. Load Dataset & Setup Transforms
test_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Using torchvision ImageFolder if your data is split into folders (e.g., data/val/0, data/val/1...)
# If you prefer using a CSV, use a custom dataset class instead.
from torchvision.datasets import ImageFolder
# Point this to your validation folder directory containing class subfolders (0, 1, 2, 3, 4)
val_dir = '/kaggle/input/APTOS 2019 Blindness Detection/train_images' # Change to your val split dir if available

# If using a folder structure with subfolders for each class:
test_dataset = ImageFolder('data/val', transform=test_transforms) 
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# 3. Define Model Class Architecture
class HybridEfficientNetMambaUpgraded(nn.Module):
    def __init__(self, num_classes=5):
        super(HybridEfficientNetMambaUpgraded, self).__init__()
        
        self.backbone = timm.create_model('efficientnet_b3', pretrained=False, features_only=True)
        
        dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE)
        features = self.backbone(dummy_input)
        final_channels = features[-1].shape[1]
        
        self.proj = nn.Conv2d(final_channels, 256, kernel_size=1)
        self.depthwise_conv = nn.Conv1d(256, 256, kernel_size=3, padding=1, groups=256)
        self.fc1 = nn.Linear(256, 512)
        self.fc2 = nn.Linear(512, 256)
        self.dropout = nn.Dropout(0.3)
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        feat_list = self.backbone(x)
        x = feat_list[-1] 
        
        x = self.proj(x)
        b, c, h, w = x.shape
        
        x = x.view(b, c, h * w) 
        x_res = x
        
        x = self.depthwise_conv(x)
        x = torch.nn.functional.silu(x)
        
        x = x.permute(0, 2, 1) 
        x = self.fc1(x)
        x = torch.nn.functional.silu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        
        x = x.permute(0, 2, 1)
        x = x + x_res
        
        out = self.classifier(x)
        return out

# 4. Load Model Weights
print(f"Loading weights from {WEIGHT_PATH}...")
model = HybridEfficientNetMambaUpgraded(num_classes=NUM_CLASSES)
model.load_state_dict(torch.load(WEIGHT_PATH, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

# 5. Run Evaluation Loop
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Evaluating Labeled Set"):
        images = images.to(DEVICE)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# 6. Calculate Metrics
accuracy = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted', zero_division=0)

print("\n================ EVALUATION METRICS ===============s")
print(f"Accuracy  : {accuracy * 100:.2f}%")
print(f"Precision : {precision * 100:.2f}%")
print(f"Recall    : {recall * 100:.2f}%")
print(f"F1-Score  : {f1 * 100:.2f}%")
print("=====================================================\n")

class_names = ['No DR (0)', 'Mild (1)', 'Moderate (2)', 'Severe (3)', 'Proliferative (4)']
print("Detailed Classification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

Loading weights from hybrid_mamba_improved_best.pth...


Evaluating Labeled Set: 100%|██████████| 46/46 [00:43<00:00,  1.06it/s]


================ EVALUATION METRICS ===============s
Accuracy  : 82.95%
Precision : 82.50%
Recall    : 82.95%
F1-Score  : 82.00%

Detailed Classification Report:
                   precision    recall  f1-score   support

        No DR (0)       0.98      0.98      0.98       361
         Mild (1)       0.58      0.57      0.57        74
     Moderate (2)       0.72      0.88      0.79       200
       Severe (3)       0.50      0.23      0.32        39
Proliferative (4)       0.76      0.49      0.60        59

         accuracy                           0.83       733
        macro avg       0.71      0.63      0.65       733
     weighted avg       0.83      0.83      0.82       733


Confusion Matrix:
[[352   8   1   0   0]
 [  6  42  25   0   1]
 [  0  19 176   3   2]
 [  1   0  23   9   6]
 [  0   4  20   6  29]]


In [13]:
import os
import pandas as pd

# Automatically search for test.csv in the kaggle input directory
excel_path = None
test_images_dir = None

for root, dirs, files in os.walk('/kaggle/input'):
    if 'test.csv' in files:
        excel_path = os.path.join(root, 'test.csv')
    if 'test_images' in dirs:
        test_images_dir = os.path.join(root, 'test_images')

print(f"Found CSV path: {excel_path}")
print(f"Found Images path: {test_images_dir}")

# Load test data safely
test_df = pd.read_csv(excel_path)
test_df['file_name'] = test_df['id_code'].astype(str) + '.png'

Found CSV path: /kaggle/input/competitions/aptos2019-blindness-detection/test.csv
Found Images path: /kaggle/input/competitions/aptos2019-blindness-detection/test_images
